# EpiAI 一站式教程

从原始数据到部署预测的完整流程：
1. 加载数据
2. 批量训练多种模型 + 排行榜
3. 部署 + 风险评分
4. 预测（含置信区间 + 多风险算法）
5. 新数据追加 + 选择性重训
6. 模型持久化与加载

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from pathlib import Path

from EpiAI import ForecastTask

## 1. 加载数据

In [ ]:
DATA_PATH = Path.home() / "infective_disease" / "pingtai" / "merged_clean.csv"
DISEASE = "登革热 Dengue fever"

df_raw = pd.read_csv(DATA_PATH)
df_raw["Year/Month"] = pd.to_datetime(df_raw["Year/Month"])
df = (
    df_raw.query("Diseases == @DISEASE")
    [["Year/Month", "Case number"]]
    .rename(columns={"Year/Month": "time", "Case number": "cases"})
    .reset_index(drop=True)
)
df = df[df["time"].dt.year >= 2010].reset_index(drop=True)

print(f"数据: {len(df)} 行, {df['time'].min().date()} → {df['time'].max().date()}")
df.head()

## 2. 定义预测任务

`ForecastTask` 是一切的核心：指定数据、目标列、频率、预测步数。

In [ ]:
task = ForecastTask(
    data=df,
    target="cases",
    time_col="time",
    freq="MS",          # 月数据
    horizon=12,          # 预测未来 12 期
)
print(f"Task: {task}")
print(f"季节周期: {task.freq_to_season_length()} 期/年")

## 3. 批量训练 + 排行榜

传入多个模型名，自动训练并返回 `ModelVault`（含对比指标）。

In [ ]:
vault = task.compare([
    "sf_auto_ets",       # StatsForecast AutoETS
    "sf_holt_winters",   # StatsForecast Holt-Winters
    "sf_theta",          # StatsForecast Theta
    "prophet",           # Facebook Prophet
    "serfling",          # Serfling 回归（传染病基线）
])

print("\n排行榜（按 R² 排序）:")
display(vault.summary())

## 4. 部署 + 风险评分

选择最佳模型部署。可以同时启用多个风险算法。

In [ ]:
best_model = vault.best("R2")
print(f"最佳模型: {best_model}")

runtime = task.deploy(
    best_model,
    vault,
    risk=["quantile", "zscore"],  # 同时用分位数和 Z-score 评分
)
print(f"Runtime: {runtime}")
print(f"风险算法: {list(runtime.risk_scorers.keys())}")

## 5. 预测

支持置信区间（`level` 参数）和多种风险评分。

In [ ]:
report = runtime.predict(horizon=6, level=95)
print("输出列:", report.columns.tolist())
display(report)

### 说明

| 列 | 含义 |
|----|------|
| `time` | 预测时间点 |
| `{model}` | 预测值（均值） |
| `{model}_lo` | 95% 置信区间下限 |
| `{model}_hi` | 95% 置信区间上限 |
| `{model}_quantile` | 分位数法风险等级 0-3 |
| `{model}_zscore` | Z-score 法风险等级 0-3 |

风险等级：0=低 🟢 1=中 🟡 2=高 🟠 3=极高 🔴

## 6. 新数据接收 + 选择性重训

模拟新数据到达：先 `feed` 追加，再 `refit` 重训。

In [ ]:
# 模拟新数据
np.random.seed(42)
new_data = pd.DataFrame({
    "time": pd.date_range("2026-05-01", periods=3, freq="MS"),
    "cases": np.random.randn(3) * 300 + 500,
})

runtime.feed(new_data)
print(f"data_table 现有 {len(runtime.data_table)} 行")

# 选择性重训（只重训 AutoETS，Holt-Winters 保持不变）
runtime.refit(model_names=["sf_auto_ets"], verbose=True)

print("\n重训后预测:")
display(runtime.predict(horizon=6))

## 7. 持久化与加载

保存到磁盘 → 服务器加载 → 恢复预测。

In [ ]:
# 保存
save_path = "/tmp/epiai_tutorial_runtime"
runtime.save(save_path)
print(f"已保存至 {save_path}")
print(f"目录内容: {list(Path(save_path).iterdir())}")

In [ ]:
# 服务器加载
from EpiAI.inference import DeploymentRuntime

loaded = DeploymentRuntime.load(save_path)
print(f"加载完成: {loaded}")
print(f"风险算法: {list(loaded.risk_scorers.keys())}")

# 恢复预测
report = loaded.predict(horizon=6, level=95)
display(report)

## 总结

```python
from EpiAI import ForecastTask

# 训练 + 对比
task = ForecastTask(data=df, target="cases", freq="MS", horizon=12)
vault = task.compare(["sf_auto_ets", "sf_holt_winters", "prophet"])

# 部署
runtime = task.deploy(vault.best("R2"), vault, risk=["quantile", "zscore"])

# 生产
report = runtime.predict(horizon=6, level=95)
runtime.feed(new_data)
runtime.refit(model_names=["sf_auto_ets"])
runtime.save("/data/runtime/")
```